# 00. ELT 파이프라인 — 코스메틱 이커머스 이벤트 로그

## 분석 개요

- **목적**: 원본 CSV 5개의 파일·스키마를 확인하고, MySQL `events` 적재 상태와 필수 인덱스를 교차 확인한다.
- **분석 단위**: 이벤트(원본 1행 = 1이벤트)
- **관측 기간**: 2019-10-01 - 2020-02-29
- **안전 원칙**: 기본 실행은 기존 DB를 읽기 전용으로 재사용한다. 명시적인 이중 확인 없이는 테이블 재생성·적재·인덱스 변경을 실행하지 않는다.
- **후속 단계**: 이벤트 품질과 전처리 방침은 `01_raw_eda.ipynb`에서 진단한다.

In [1]:
import os
import re
import time
from pathlib import Path
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, event, text

PROJECT_ROOT = next(
    (base for base in (Path.cwd(), Path.cwd().parent) if (base / '.env').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('프로젝트 루트의 .env를 찾을 수 없다.')

load_dotenv(PROJECT_ROOT / '.env')
data_dir_value = os.getenv('DATA_DIR')
configured_data_dir = Path(data_dir_value).expanduser() if data_dir_value else None
DATA_DIR = (
    configured_data_dir
    if configured_data_dir is not None and configured_data_dir.is_dir()
    else PROJECT_ROOT / 'data'
)

FILES = ['2019-Oct.csv', '2019-Nov.csv', '2019-Dec.csv', '2020-Jan.csv', '2020-Feb.csv']
REQUIRED_COLUMNS = [
    'event_time', 'event_type', 'product_id', 'category_id', 'category_code',
    'brand', 'price', 'user_id', 'user_session',
]

# 두 값을 모두 명시적으로 바꾼 경우에만 events를 재생성한다.
REBUILD_EVENTS = False
REBUILD_CONFIRMATION = ''
REBUILD_ALLOWED = (
    REBUILD_EVENTS is True
    and REBUILD_CONFIRMATION == 'DROP_AND_RELOAD_EVENTS'
)

# 계산값이 아니라 현재 데이터셋의 회귀 참고값이다.
REGRESSION_TOTAL = 20_692_840
REGRESSION_START = pd.Timestamp('2019-10-01')
REGRESSION_END_DATE = pd.Timestamp('2020-02-29').date()

---
## 1. 원본 파일과 스키마 확인

월별 CSV 5개의 존재 여부, 파일명에서 해석한 월 범위, 공통 헤더와 샘플 자료형만 확인한다. 전체 CSV를 메모리에 올리거나 품질 EDA를 수행하지 않는다.

In [2]:
missing_files = [name for name in FILES if not (DATA_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(f'원본 CSV가 없다: {missing_files} (DATA_DIR={DATA_DIR})')

sample_frames = {name: pd.read_csv(DATA_DIR / name, nrows=5) for name in FILES}
invalid_headers = {
    name: frame.columns.tolist()
    for name, frame in sample_frames.items()
    if frame.columns.tolist() != REQUIRED_COLUMNS
}
if invalid_headers:
    raise ValueError(f'원본 CSV 헤더가 예상 스키마와 다르다: {invalid_headers}')

file_manifest = pd.DataFrame({
    '파일': FILES,
    '월': [pd.to_datetime(name[:8], format='%Y-%b').strftime('%Y-%m') for name in FILES],
    '존재': [(DATA_DIR / name).is_file() for name in FILES],
    '크기_MB': [round((DATA_DIR / name).stat().st_size / 1024**2, 1) for name in FILES],
    '헤더_일치': [sample_frames[name].columns.tolist() == REQUIRED_COLUMNS for name in FILES],
})
source_schema = pd.DataFrame({
    '컬럼': REQUIRED_COLUMNS,
    '샘플_자료형': sample_frames[FILES[0]].dtypes.astype(str).reindex(REQUIRED_COLUMNS).values,
})
display(file_manifest)
source_schema

,파일,월,존재,크기_MB,헤더_일치
0,2019-Oct.csv,2019-10,True,460.2,True
1,2019-Nov.csv,2019-11,True,520.6,True
2,2019-Dec.csv,2019-12,True,396.1,True
3,2020-Jan.csv,2020-01,True,478.5,True
4,2020-Feb.csv,2020-02,True,466.2,True


,컬럼,샘플_자료형
0,event_time,object
1,event_type,object
2,product_id,int64
3,category_id,int64
4,category_code,float64
5,brand,object
6,price,float64
7,user_id,int64
8,user_session,object


---
## 2. 안전 모드와 DB 연결

`REBUILD_EVENTS=False`, `REBUILD_CONFIRMATION=''`가 기본값이다. 두 조건을 모두 충족하지 않으면 `DROP`, `CREATE`, CSV 적재, 인덱스 생성·변경을 실행하지 않는다. 기존 `events`가 없을 때도 자동으로 만들지 않고 재생성 승인 방법을 안내하며 중단한다.

In [3]:
required_env = ['DB_USER', 'DB_PASSWORD', 'DB_HOST', 'DB_PORT', 'DB_NAME']
missing_env = [key for key in required_env if not os.getenv(key)]
if missing_env:
    raise EnvironmentError(f'.env에 DB 접속 변수가 없다: {missing_env}')

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{quote_plus(os.getenv('DB_PASSWORD'))}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}?charset=utf8mb4",
    connect_args={'local_infile': 1},
)
executed_sql_types = []

@event.listens_for(engine, 'before_cursor_execute')
def _record_sql_type(conn, cursor, statement, parameters, context, executemany):
    match = re.match(r'^\s*(?:/\*.*?\*/\s*)?(\w+)', statement, flags=re.DOTALL)
    executed_sql_types.append(match.group(1).upper() if match else 'UNKNOWN')

with engine.connect() as conn:
    events_exists = conn.execute(text("SHOW TABLES LIKE 'events'")).first() is not None

if not events_exists and not REBUILD_ALLOWED:
    raise RuntimeError(
        "events 테이블이 없다. 재생성이 필요하면 REBUILD_EVENTS=True와 "
        "REBUILD_CONFIRMATION='DROP_AND_RELOAD_EVENTS'를 함께 설정해야 한다."
    )

pd.DataFrame([{
    'DB': os.getenv('DB_NAME'),
    'events_존재': events_exists,
    '실행_모드': 'rebuild' if REBUILD_ALLOWED else 'read-only reuse',
}])

,DB,events_존재,실행_모드
0,Cosmetics_Funnel,True,read-only reuse


---
## 3. 승인된 경우에만 재생성

재생성 모드는 원본 파일·헤더 확인을 통과한 뒤 `events`를 초기화하고 월별 CSV를 순차 적재한다. 인덱스 정의는 수정하지 않는 `sql/indexes.sql`을 단일 원천으로 사용한다. 이번 기본 실행에서는 이 분기 전체를 건너뛴다.

In [4]:
CREATE_EVENTS = """
CREATE TABLE events (
    event_time    DATETIME     NOT NULL,
    event_type    VARCHAR(20)  NOT NULL,
    product_id    BIGINT       NOT NULL,
    category_id   BIGINT,
    category_code VARCHAR(120),
    brand         VARCHAR(60),
    price         DECIMAL(10, 2),
    user_id       BIGINT       NOT NULL,
    user_session  CHAR(36)
)
"""
LOAD_EVENTS = """
LOAD DATA LOCAL INFILE :fpath
INTO TABLE events
FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"'
LINES TERMINATED BY '\n'
IGNORE 1 LINES
(@event_time, event_type, product_id, category_id, @category_code, @brand, price, user_id, @user_session)
SET
    event_time = STR_TO_DATE(REPLACE(@event_time, ' UTC', ''), '%Y-%m-%d %H:%i:%s'),
    category_code = NULLIF(@category_code, ''),
    brand = NULLIF(@brand, ''),
    user_session = NULLIF(@user_session, '')
"""

def _load_named_queries(path):
    body = Path(path).read_text(encoding='utf-8')
    parts = re.split(r'(?m)^--\s*name:\s*(\w+).*$', body)
    return {parts[i]: parts[i + 1].strip() for i in range(1, len(parts), 2)}

INDEX_SQL_FILE = PROJECT_ROOT / 'sql' / 'indexes.sql'
index_queries = _load_named_queries(INDEX_SQL_FILE)
if not index_queries:
    raise ValueError(f'필수 인덱스 정의를 찾을 수 없다: {INDEX_SQL_FILE}')

In [5]:
rebuild_log = []
if REBUILD_ALLOWED:
    with engine.begin() as conn:
        conn.execute(text('DROP TABLE IF EXISTS events'))
        conn.execute(text(CREATE_EVENTS))

    for name in FILES:
        started_at = time.perf_counter()
        with engine.begin() as conn:
            result = conn.execute(text(LOAD_EVENTS), {'fpath': str((DATA_DIR / name).resolve())})
            warnings = conn.execute(text('SHOW WARNINGS')).fetchall()
        rebuild_log.append({
            '파일': name,
            '적재행수': result.rowcount,
            '경고': len(warnings),
            '소요초': round(time.perf_counter() - started_at, 1),
        })

    with engine.begin() as conn:
        for name, statement in index_queries.items():
            conn.execute(text(statement))
    rebuild_status = pd.DataFrame(rebuild_log)
else:
    rebuild_status = pd.DataFrame([{
        '상태': '건너뜀',
        '사유': '이중 안전 조건 미충족 — 기존 events를 읽기 전용으로 재사용',
    }])

rebuild_status

,상태,사유
0,건너뜀,이중 안전 조건 미충족 — 기존 events를 읽기 전용으로 재사용


---
## 4. 적재 완결성 교차 확인

전체·월별 행 수, 관측 시작·종료 시각, 필수 컬럼과 인덱스만 확인한다. 20,692,840행과 2019-10-01 - 2020-02-29는 계산값을 대체하지 않는 이전 실행 기준값으로만 비교한다.

In [6]:
event_summary = pd.read_sql(text("""
    SELECT
        COUNT(*) AS total_rows,
        MIN(event_time) AS observed_start,
        MAX(event_time) AS observed_end
    FROM events
"""), engine)
monthly_rows = pd.read_sql(text("""
    SELECT
        DATE_FORMAT(event_time, '%Y-%m') AS month,
        COUNT(*) AS row_count
    FROM events
    GROUP BY DATE_FORMAT(event_time, '%Y-%m')
    ORDER BY month
"""), engine)

total_rows = int(event_summary.loc[0, 'total_rows'])
observed_start = pd.Timestamp(event_summary.loc[0, 'observed_start'])
observed_end = pd.Timestamp(event_summary.loc[0, 'observed_end'])
monthly_total = int(monthly_rows['row_count'].sum())
expected_months = [pd.to_datetime(name[:8], format='%Y-%b').strftime('%Y-%m') for name in FILES]

completeness = pd.DataFrame([{
    '전체_행수': total_rows,
    '월별_합계': monthly_total,
    '합계_일치': total_rows == monthly_total,
    '관측_시작': observed_start,
    '관측_종료': observed_end,
    '5개월_완결': monthly_rows['month'].tolist() == expected_months,
    '회귀_행수_일치': total_rows == REGRESSION_TOTAL,
    '회귀_기간_일치': (
        observed_start == REGRESSION_START
        and observed_end.date() == REGRESSION_END_DATE
    ),
}])
if not bool(completeness.loc[0, '합계_일치']) or not bool(completeness.loc[0, '5개월_완결']):
    raise AssertionError('events 전체 행 수와 월별 완결성 검산에 실패했다.')

display(completeness)
monthly_rows

,전체_행수,월별_합계,합계_일치,관측_시작,관측_종료,5개월_완결,회귀_행수_일치,회귀_기간_일치
0,20692840,20692840,True,2019-10-01,2020-02-29 23:59:59,True,True,True


,month,row_count
0,2019-10,4102283
1,2019-11,4635837
2,2019-12,3533286
3,2020-01,4264752
4,2020-02,4156682


In [7]:
columns = pd.read_sql(text('SHOW COLUMNS FROM events'), engine)
indexes = pd.read_sql(text('SHOW INDEX FROM events'), engine)

actual_columns = columns['Field'].tolist()
expected_index_columns = {}
for name, statement in index_queries.items():
    match = re.search(r'ON\s+events\s*\((.*?)\)', statement, flags=re.IGNORECASE | re.DOTALL)
    if match is None:
        raise ValueError(f'{name} 인덱스 컬럼 정의를 해석할 수 없다.')
    expected_index_columns[name] = tuple(
        part.strip().strip('`').split()[0]
        for part in match.group(1).split(',')
    )
actual_index_columns = (
    indexes[indexes['Key_name'].isin(index_queries)]
    .sort_values(['Key_name', 'Seq_in_index'])
    .groupby('Key_name')['Column_name']
    .apply(tuple)
    .to_dict()
)
index_check = pd.DataFrame([
    {
        '인덱스': name,
        '기대_컬럼': ', '.join(expected),
        '실제_컬럼': ', '.join(actual_index_columns.get(name, ())),
        '일치': actual_index_columns.get(name) == expected,
    }
    for name, expected in expected_index_columns.items()
])
schema_ok = actual_columns == REQUIRED_COLUMNS
indexes_ok = bool(index_check['일치'].all())
if not schema_ok or not indexes_ok:
    raise AssertionError('events 필수 컬럼 또는 인덱스 검산에 실패했다.')

if not REBUILD_ALLOWED:
    unexpected_sql_types = sorted(set(executed_sql_types) - {'SELECT', 'SHOW'})
    if unexpected_sql_types:
        raise AssertionError(f'안전 모드에서 쓰기 가능 SQL이 실행됐다: {unexpected_sql_types}')

execution_audit = pd.DataFrame([{
    '필수_컬럼_일치': schema_ok,
    '필수_인덱스_일치': indexes_ok,
    '실행_SQL_유형': ', '.join(sorted(set(executed_sql_types))),
    '안전_모드_쓰기_0건': (
        not REBUILD_ALLOWED
        and set(executed_sql_types).issubset({'SELECT', 'SHOW'})
    ),
}])
display(execution_audit)
index_check

,필수_컬럼_일치,필수_인덱스_일치,실행_SQL_유형,안전_모드_쓰기_0건
0,True,True,"SELECT, SHOW",True


,인덱스,기대_컬럼,실제_컬럼,일치
0,idx_user_time,"user_id, event_time","user_id, event_time",True
1,idx_product,product_id,product_id,True
2,idx_event_type,event_type,event_type,True
3,idx_repeat_events,"user_session, event_time, product_id, event_type","user_session, event_time, product_id, event_type",True


---
## 5. 결론과 다음 단계

> ### 적재 확인 결론
>
> - 원본 5개월 파일을 `events`에 적재하고 필수 인덱스를 확인했다.
> - 이번 기본 실행은 기존 `events`를 읽기 전용으로 재사용했으며 DB 쓰기를 수행하지 않았다.
> - 이벤트 품질과 전처리 방침은 `01_raw_eda.ipynb`에서 진단한다.